<a href="https://colab.research.google.com/github/AnkaRezaa/machine_learning_254107023002/blob/main/jobsheet%20dataset%20AntiBurnOut/JS03/03_Feature_Engineering_Encoding_Scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS03: Feature Engineering, Encoding & Scaling - Dataset AntiBurnOut

**Identitas Mahasiswa:**
- **Nama**  : MUHAMMAD ANKA REZA
- **NIM**   : 254107023002
- **Kelas** : TI/3H

---

### Deskripsi Praktikum:
Praktikum ini berfokus pada tahapan rekayasa fitur (*feature engineering*), encoding variabel teks/kategorikal (One-Hot Encoding untuk nominal dan Ordinal Mapping untuk tingkatan ordinal), normalisasi fitur numerik dengan `StandardScaler`, serta pembagian dataset menjadi data latih (*train*), validasi (*validation*), dan uji (*test*) dengan stratifikasi target.

### Langkah 1: Memuat Data dan Pra-Pengolahan Nilai Kosong

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

dataset_candidates = [
    'mental_health_burnout_prediction_dataset.csv',
    '../mental_health_burnout_prediction_dataset.csv',
    'jobsheet dataset AntiBurnOut/mental_health_burnout_prediction_dataset.csv'
]
dataset_path = next((p for p in dataset_candidates if os.path.exists(p)), 'mental_health_burnout_prediction_dataset.csv')
df = pd.read_csv(dataset_path)

# Hapus ID jika ada
if 'Person_ID' in df.columns:
    df = df.drop(columns=['Person_ID'])

# Imputasi cepat
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Dimensi data awal:", df.shape)

### Langkah 2: Encoding Variabel Teks / Kategorik
Menerapkan Ordinal Mapping untuk fitur bertingkat dan One-Hot Encoding untuk fitur teks nominal.

In [ ]:
# 1. Ordinal Mapping
sleep_map = {'Poor': 0, 'Average': 1, 'Good': 2, 'Excellent': 3}
stress_map = {'Low': 0, 'Moderate': 1, 'High': 2}
exercise_map = {'Never': 0, 'Rarely': 1, 'Weekly': 2, 'Daily': 3}
support_map = {'Poor': 0, 'Average': 1, 'Good': 2}

df['Sleep_Quality_Ord'] = df['Sleep_Quality'].map(sleep_map).fillna(1)
df['Stress_Level_Ord'] = df['Stress_Level'].map(stress_map).fillna(1)
df['Exercise_Frequency_Ord'] = df['Exercise_Frequency'].map(exercise_map).fillna(1)
df['Support_System_Ord'] = df['Support_System'].map(support_map).fillna(1)

# 2. Pemetaan Biner
bin_cols = ['Remote_Work', 'Alcohol_Consumption', 'Smoking', 'Healthy_Diet', 
            'Chronic_Stress', 'Family_History_Mental_Illness', 'Therapy_Attendance']
for b in bin_cols:
    df[b + '_Bin'] = df[b].apply(lambda x: 1 if str(x).strip().lower() in ['yes', 'hybrid', 'true', '1'] else 0)

# 3. One-Hot Encoding untuk Nominal (Gender, Occupation, Education_Level, Employment_Status)
nom_cols = ['Gender', 'Education_Level', 'Employment_Status']
df_encoded = pd.get_dummies(df, columns=nom_cols, drop_first=True)

print("Jumlah kolom setelah encoding:", df_encoded.shape[1])

### Langkah 3: Separasi Fitur (X) dan Target (y)

In [ ]:
# Target Klasifikasi: Burnout_Risk
# Target Regresi: Burnout_Score
y_class = df_encoded['Burnout_Risk']
y_reg = df_encoded['Burnout_Score']

# Drop kolom target dan kolom teks mentah yang sudah di-encode
drop_targets = ['Burnout_Risk', 'Burnout_Score', 'Mental_Health_Status', 'AI_Wellness_Recommendation',
                'Sleep_Quality', 'Stress_Level', 'Exercise_Frequency', 'Support_System',
                'Country', 'Occupation'] + bin_cols

X = df_encoded.drop(columns=[c for c in drop_targets if c in df_encoded.columns])
print("Fitur input (X) siap diproses. Shape:", X.shape)

### Langkah 4: Pembagian Data (Train, Validation, Test Split) & Standarisasi

In [ ]:
# Pembagian 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y_class, test_size=0.3, random_state=42, stratify=y_class)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Standarisasi Fitur Numerik menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Data Latih      : {X_train_scaled.shape}")
print(f"Data Validasi   : {X_val_scaled.shape}")
print(f"Data Pengujian  : {X_test_scaled.shape}")

### Kesimpulan JS03:
1. Seluruh variabel teks dan kategorik berhasil di-transformasi menjadi bentuk matriks numerik yang siap diproses oleh algoritma Machine Learning.
2. Pembagian data latih, validasi, dan uji dilakukan dengan proporsi yang seimbang (*stratified*) sehingga distribusi kelas risiko burnout dipertahankan secara proporsional.